# Lecture 1 · Notebook 3 — One model, many tasks
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture01/lecture01_3_one_model_many_tasks.ipynb)

**Companion to [Chapter 5 of the video lecture](https://nlp-course-cuhksz.github.io/lecture01.html#p5-s67)** (slides 59–67, the classic tasks) and
[Chapter 7](https://nlp-course-cuhksz.github.io/lecture01.html#p7-s85) (slides 85–88, large language models). The lecture's key message on slide 67: once a model really works with
language, **language becomes a universal interface**. Each classic task used to need its own system; here one small instruction-tuned
model (Qwen2.5-0.5B-Instruct, 0.5B parameters) does them all from a prompt. You will:

1. compare a **keyword lexicon** with the LLM on tricky sentiment (sarcasm, negation; slide 62),
2. run translation, classification, question answering and slot extraction with the **same model** (slides 60–66),
3. look under the hood: the answer is still just **next-token probabilities** (slide 80).

Runs on CPU in 5–10 minutes (~1 GB download). A T4 GPU makes it faster but is not needed. This is a *tiny* model: it makes mistakes,
which is part of the lesson. Frontier models are hundreds of times larger (slide 88).

In [ ]:
!pip -q install -U transformers accelerate

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
llm = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16 if device == "cuda" else torch.float32).to(device).eval()
print(f"{MODEL_ID}: {sum(p.numel() for p in llm.parameters()) / 1e6:.0f}M parameters on {device}")

@torch.no_grad()
def ask(prompt, max_new_tokens=60, system="You are a helpful assistant. Answer briefly."):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(device)
    out = llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,      # greedy: same answer every run
                       temperature=None, top_p=None, top_k=None)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(ask("In one sentence, what is natural language processing?"))

## 1 · Sentiment: a keyword lexicon vs. the LLM
A classic baseline counts positive and negative words. Slide 62 warns that sarcasm, mixed feelings and negation fool it.

In [ ]:
POS = {"great", "love", "good", "excellent", "amazing", "best", "happy", "perfect", "nice"}
NEG = {"bad", "terrible", "hate", "awful", "worst", "delayed", "broken", "sad", "angry"}
def lexicon_sentiment(text):
    w = [x.strip(".,!?'\"").lower() for x in text.split()]
    s = sum(x in POS for x in w) - sum(x in NEG for x in w)
    return "positive" if s > 0 else "negative" if s < 0 else "neutral"

reviews = [
    "I love this coffee, best latte in town.",
    "Oh great, my order is late again. Just perfect.",
    "Not bad at all, I would come back.",
    "The barista was nice but the coffee was terrible.",
    "My phone is broken, so I'm writing this review from my laptop: the new store is amazing.",
]
def llm_sentiment(text):
    a = ask(f"Classify the sentiment of this review as positive, negative or mixed. Reply with one word.\nReview: {text}", max_new_tokens=4)
    return a.lower().strip(".")

for r in reviews:
    print(f"{r[:62]:62s} lexicon={lexicon_sentiment(r):9s} llm={llm_sentiment(r)}")

Count where each one fails. The lexicon has no idea that *"Oh great ... just perfect"* is sarcastic, that *"not bad"* is positive,
or that the broken phone is not about the store. The LLM reads the whole sentence, but a 0.5B model is not always right either: in our
test run it called the sarcastic review *mixed* rather than *negative* (section 4 shows how close that decision was).

## 2 · Many tasks, one model, just change the prompt
Each block below used to be a separate research area with its own models and datasets (slides 60–66).

In [ ]:
tasks = {
    "Machine translation (slide 60)":
        "Translate into English: buenas noches, y gracias por todo.",
    "Translation into Chinese":
        "Translate into Chinese: Language is how humans store and share knowledge.",
    "Text classification (slide 63)":
        "Which Gmail tab fits this email best: Primary, Social, Promotions or Updates? Reply with one word.\n"
        "Email: 50% off all shoes this weekend only! Use code SUMMER50.",
    "Forgot the attachment? (slide 63)":
        "Does this email say a file is attached? Answer yes or no.\nEmail: Hi Prof. Wang, my report is attached. Best, Li",
    "Question answering with a context (slide 65)":
        "Context: Cheyenne is the capital and most populous city of Wyoming.\nQuestion: What is the capital of Wyoming?",
}
for name, prompt in tasks.items():
    print(f"## {name}\n{ask(prompt)}\n")

## 3 · Slot extraction without a pipeline
Notebook 1 needed a tokenizer, a tagger, a parser, an entity recognizer and hand-written rules for this (slide 68). Here we simply ask for JSON,
including the lower-case, misspelled version that broke the pipeline.

In [ ]:
import json, re
def extract(text):
    a = ask("Extract the slots from the user's request as JSON with keys action, cuisine, location, time "
            "(use null if missing). Reply with JSON only.\nRequest: " + text, max_new_tokens=80)
    m = re.search(r"\{.*\}", a, re.S)
    try:
        return json.loads(m.group(0)) if m else a
    except json.JSONDecodeError:
        return a

for text in ["Find me an Italian restaurant in New York City.",
             "find me an itallian restaurant in new yrok",
             "Set an alarm for 7 tomorrow and add gelato to my shopping list."]:
    print(f"{text:66s} -> {extract(text)}")

Compare with Notebook 1: no tokenizer, tagger, parser or rules, and the misspelled cuisine comes back as *Italian*. But read the output
critically. In our test run the small model dropped the misspelled city, and for *"tomorrow"* it may invent a concrete date it cannot know,
a first example of **hallucination**. Larger models make such mistakes less often, not never.

## 4 · Under the hood it is still next-token prediction
The model does not have a sentiment module or a translation module. For each task it only computes a probability distribution over the
next token, exactly the conditional language model of slide 80. Here is the distribution for the first token of the sarcastic review's label.
Note that *Negative* and *negative* are different tokens: the probability of "negative" is split between them.

In [ ]:
@torch.no_grad()
def first_token_probs(prompt, k=6):
    msgs = [{"role": "system", "content": "You are a helpful assistant. Answer briefly."}, {"role": "user", "content": prompt}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(device)
    p = llm(**enc).logits[0, -1].float().softmax(-1)
    top = p.topk(k)
    return [(tok.decode(i), round(v.item(), 3)) for v, i in zip(top.values, top.indices)]

q = "Classify the sentiment of this review as positive, negative or mixed. Reply with one word.\nReview: " + reviews[1]
print(first_token_probs(q))

## Exercises
1. Write three more reviews that fool the lexicon. Does the LLM get them right? Does adding *"Watch out for sarcasm."* to the prompt help?
2. Add a task of your own to section 2 (summarize a paragraph, detect toxic language, route a support ticket). Which tasks does the
   0.5B model still fail, and would you fix them with a better prompt, a larger model, or fine-tuning (the question on slide 31)?
3. If you have a GPU, change `MODEL_ID` to `Qwen/Qwen2.5-1.5B-Instruct` and rerun. What does three times more parameters buy you?

**Back to the [video lecture]({SITE})**